In [50]:
import math
import random

In [47]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
            self.data + other.data,
            (self, other),
            '+'
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __mul__(self, other):
        other =  other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data * other.data,
        (self, other),
        '*'

    )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out
    def __rmul__(self, other):
        return self * other

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(
        self.data - other.data,
        (self, other),
        '-'
    )

        def _backward():
            self.grad += out.grad
            other.grad -= out.grad

        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other * self**-1
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


In [ ]:
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)] 
        self.b = Value(0)
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

In [33]:
c = Value(2.0) + Value(3.0)
c.data

5.0

In [34]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

result = a * b + c


result.backward()

print(result.data)
print(a.grad)
print(b.grad)
print(c.grad)

10.0
3.0
2.0
1.0


In [35]:
a = Value(5.0)
b = a * 3

print(b.data)  # 15.0

15.0


In [36]:
a = Value(2.0)
b = Value(3.0)

c = a * b + a * b + 5
c.backward()

print(c.data)
print(a.grad)
print(b.grad)

17.0
6.0
4.0


In [40]:
a = Value(3.0)
b = a**2

b.backward()
print(b.data)
print(a.grad)

9.0
6.0


In [41]:
a = Value(10.0)
b = Value(2.0)

c = a / b

c.backward()

print(c.data)
print(a.grad)
print(b.grad)

5.0
0.5
-2.5


In [45]:
a = Value(0.5)
b = a.tanh()

b.backward()

print(b.data)
print(a.grad)

0.46211715726000974
0.7864477329659274


In [49]:
a = Value(4.0)
b = a.relu()

b.backward()

print(b.data)
print(a.grad)

4.0
1.0
